# AB-DB tutorial: finding, downloading, visualizing and retrieving analysis
***

This Jupyter Notebook will guide you through the process of **finding**, **downloading**, **visualizing** and finally **retrieving analysis results** from the collection of **AB-DB** ligands **MD simulations** and **QM data** available in the **[AB-DB](https://mmb.irbbarcelona.org/ABDB/) database**.

This workflow is based on the **AB-DB database REST API**: https://mmb.irbbarcelona.org/ABDB/rest

Main **AB-DB website**: https://mmb.irbbarcelona.org/ABDB/

***Authors:*** *Giuliano Malloci, Silvia Gervasoni, Adam Hospital, Genís Bayarri*
<br>
***Version:*** *July 2026*

***
## Tutorial steps
 1. [Libraries and Global Variables](#libraries)
 2. [Listing AB-DB Compounds](#listing)
 3. [Browsing AB-DB Database](#browsing)
 4. [Searching & Finding in the AB-DB Database](#searching)
 5. [Physico-chemical descriptors from AB-DB](#descriptors)
 6. [MD data from AB-DB](#md)
    - *Structure downloading*
    - *Structure visualization*
    - *Trajectory downloading*
    - *Trajectory visualization*
    - *Trajectory analyses (MDDB<sup>1</sup>)*
 7. [QM data from AB-DB](#qm)
    - *Download the QM-optimized structure and compare it against the original one*
    - *AMBER force-field parameters generated from the QM calculations*
    - *Geometry optimization and Single-point energy calculation data (ioChem-BD<sup>2</sup>)*
 8. [Questions & Comments](#questions)


<sup>1</sup> https://mdposit.mddbr.eu/
<br>
<sup>2</sup> https://www.iochem-bd.org/
***

<div style="text-align:center;">
<img src="https://upload.wikimedia.org/wikipedia/commons/d/de/Logo_Universit%C3%A0_di_Cagliari.jpg" alt="University of Cagliari logo"
	title="University of Cagliari logo" width="150" style="margin-right: 20px; align: center"/>
<img src="https://mmb.irbbarcelona.org/ABDB/img/home/irb.png" alt="IRB logo"
	title="IRB logo" width="250" style="margin-left: 20px;"/>
</div>

***

In [1]:
# Only executed when using google colab
import sys
import os
if 'google.colab' in sys.modules:

  try:
      from google.colab import output as _colab_output
      _colab_output.enable_custom_widget_manager()  # must precede nglview import
      get_ipython().run_line_magic('pip', 'install -q "nglview==3.0.8" mdtraj py3Dmol rich')
  except ImportError:
      pass  # not running in Colab

<a id="libraries"></a>
## Libraries and Global Variables  

In [2]:
# Auxiliary libraries

import urllib.request
import json
import os
from math import floor, ceil
from os.path import exists
from urllib.parse import quote

import ipywidgets
import nglview
#import simpletraj

import pandas as pd

#import plotly
#import plotly.graph_objs as go
#plotly.offline.init_notebook_mode(connected=True)

from rich import print
from rich.panel import Panel

In [3]:
# URLs to the REST APIs and Web Servers

web_url = 'https://mmb.irbbarcelona.org/ABDB'
api_url = f'{web_url}/api'

mddb_web = 'https://mdposit.mddbr.eu/'

iochem_bd_web = 'https://iochem-bd.org/'

<a id="listing"></a>
## Listing AB-DB Compounds
Listing the **AB-DB collection compounds**.

Exposing the **AB-DB persistent id (FAIR_ID)**, **compound**, **charge**, **family**, **formula**, **inchikey**, **pubchem** and **MDDB accession**.

In [4]:
compounds_request = f'{api_url}/compounds?limit=1000'

# Query the API
with urllib.request.urlopen(compounds_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

print(json.dumps(parsed_response['data'], indent=2))


[
  {
    "FAIR_ID": "ABDB001001",
    "Family": "aminocoumarins",
    "Compound": "chlorobiocin",
    "Formula": "C35H36ClN2O11",
    "Charge": -1,
    "accession": "MD-A007LC",
    "inchikey": "FJAQNRBDVKIIKK-LFLQOBSNSA-M",
    "pubchem": "90657643"
  },
  {
    "FAIR_ID": "ABDB001002",
    "Family": "aminocoumarins",
    "Compound": "declovanillobiocin",
    "Formula": "C31H31N2O12",
    "Charge": -1,
    "accession": "MD-A007LZ",
    "inchikey": "JURYCYSCKRZQLD-UHFFFAOYSA-N",
    "pubchem": "76173558"
  },
  {
    "FAIR_ID": "ABDB001003",
    "Family": "aminocoumarins",
    "Compound": "isovanillobiocin",
    "Formula": "C31H30ClN2O12",
    "Charge": -1,
    "accession": "MD-A007M0",
    "inchikey": "AKYRQFMQZJFLAE-UHFFFAOYSA-N",
    "pubchem": "76181969"
  },
  {
    "FAIR_ID": "ABDB001004",
    "Family": "aminocoumarins",
    "Compound": "novclobiocin_101",
    "Formula": "C35H37N2O11",
    "Charge": -1,
    "accession": "MD-A007M1",
    "inchikey": "GMMYCBAAMDJSHV-YQQRZDPSSA-M",
    "pubchem": "90659375"
  },
  {
    "FAIR_ID": "ABDB001005",
    "Family": "aminocoumarins",
    "Compound": "novobiocin",
    "Formula": "C31H35N2O11",
    "Charge": -1,
    "accession": "MD-A007M2",
    "inchikey": "DYYIUCVAQGRACE-UHFFFAOYSA-N",
    "pubchem": "4546"
  },
  {
    "FAIR_ID": "ABDB001006",
    "Family": "aminocoumarins",
    "Compound": "plazomicin",
    "Formula": "C25H53N6O10",
    "Charge": 5,
    "accession": "MD-A007M3",
    "inchikey": "IYDYFVUFSPQPPV-UHFFFAOYSA-N",
    "pubchem": "74821141"
  },
  {
    "FAIR_ID": "ABDB001007",
    "Family": "aminocoumarins",
    "Compound": "ribostamycin",
    "Formula": "C17H37N4O10",
    "Charge": 3,
    "accession": "MD-A007M4",
    "inchikey": "NSKGQURZWSPSBC-VVPCINPTSA-Q",
    "pubchem": "52358477"
  },
  {
    "FAIR_ID": "ABDB001008",
    "Family": "aminocoumarins",
    "Compound": "ribostamycin",
    "Formula": "C17H38N4O10",
    "Charge": 4,
    "accession": "MD-A007M5",
    "inchikey": "NSKGQURZWSPSBC-VVPCINPTSA-R",
    "pubchem": "51035374"
  },
  {
    "FAIR_ID": "ABDB001009",
    "Family": "aminocoumarins",
    "Compound": "vanillobiocin",
    "Formula": "C31H30ClN2O12",
    "Charge": -1,
    "accession": "MD-A007M6",
    "inchikey": "LNDDBMXOCFEELJ-UHFFFAOYSA-N",
    "pubchem": "76180016"
  },
  {
    "FAIR_ID": "ABDB002001",
    "Family": "aminoglycosides",
    "Compound": "amikacin",
    "Formula": "C22H46N5O13",
    "Charge": 3,
    "accession": "MD-A007N7",
    "inchikey": "LKCWBDHBTVXHDL-UHFFFAOYSA-N",
    "pubchem": "2142"
  },
  {
    "FAIR_ID": "ABDB002002",
    "Family": "aminoglycosides",
    "Compound": "amikacin",
    "Formula": "C22H47N5O13",
    "Charge": 4,
    "accession": "MD-A007N8",
    "inchikey": "LKCWBDHBTVXHDL-RMDFUYIESA-R",
    "pubchem": "90657305"
  },
  {
    "FAIR_ID": "ABDB002003",
    "Family": "aminoglycosides",
    "Compound": "apramycin",
    "Formula": "C21H45N5O11",
    "Charge": 4,
    "accession": "MD-A007N9",
    "inchikey": "XZNUGFQTQHRASN-UHFFFAOYSA-N",
    "pubchem": "439519"
  },
  {
    "FAIR_ID": "ABDB002004",
    "Family": "aminoglycosides",
    "Compound": "arbekacin",
    "Formula": "C22H48N6O10",
    "Charge": 4,
    "accession": "MD-A007NA",
    "inchikey": "MKKYBZZTJQGVCD-UHFFFAOYSA-N",
    "pubchem": "2226"
  },
  {
    "FAIR_ID": "ABDB002005",
    "Family": "aminoglycosides",
    "Compound": "arbekacin",
    "Formula": "C22H49N6O10",
    "Charge": 5,
    "accession": "MD-A007NB",
    "inchikey": "MKKYBZZTJQGVCD-XTCKQBCOSA-S",
    "pubchem": "90658692"
  },
  {
    "FAIR_ID": "ABDB002006",
    "Family": "aminoglycosides",
    "Compound": "dibekacin",
    "Formula": "C18H41N5O8",
    "Charge": 4,
    "accession": "MD-A007NC",
    "inchikey": "JJCQSGDBDPYCEO-UHFFFAOYSA-N",
    "pubchem": "3021"
  },
  {
    "FAIR_ID": "ABDB002007",
    "Family": "aminoglycosides",
    "Compound": "dibekacin",
    "Formula": "C18H42N5O8",
    "Charge": 5,
    "accession": "MD-A007ND",
    "inchikey": "JJCQSGDBDPYCEO-XVZSLQNASA-S",
    "pubch

<a id="browsing"></a>
## Browsing AB-DB Database

**Browse** the **AB-DB collection** from the **MDDB database**.

In [5]:
from collections import defaultdict

families = defaultdict(list)

compounds_request = f'{api_url}/compounds?limit=1000'

# Query the API
with urllib.request.urlopen(compounds_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

for entry in parsed_response['data']:
    #compound = entry['Compound']
    family = entry['Family'].title()
    families[family].append(entry)

for family, entries in families.items():
    print(f"=== {family} ===")
    for entry in entries:
        print(f"- Compound: {entry['Compound']}, Charge: {entry['Charge']}, ABDB Accession: {entry['FAIR_ID']}")
    print()


=== Aminocoumarins ===

- Compound: chlorobiocin, Charge: -1, ABDB Accession: ABDB001001

- Compound: declovanillobiocin, Charge: -1, ABDB Accession: ABDB001002

- Compound: isovanillobiocin, Charge: -1, ABDB Accession: ABDB001003

- Compound: novclobiocin_101, Charge: -1, ABDB Accession: ABDB001004

...

=== Arsenicals ===

- Compound: nitarsone, Charge: 0, ABDB Accession: ABDB050001

- Compound: roxarsone, Charge: 0, ABDB Accession: ABDB050002

=== Quinolines ===

- Compound: nitroxoline, Charge: 0, ABDB Accession: ABDB051001

In [6]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# Dropdown
dropdown = widgets.Dropdown(
    options=sorted(families.keys()),
    description="Family:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)
output = widgets.Output()

def show_class(change):
    output.clear_output()
    clase = change['new']
    with output:
        print(f"=== {clase} ===\n")
        for entry in families[clase]:
            print(f"--- {entry['FAIR_ID']} ---")
            print(f"{entry['Compound'].title()} {entry['Charge']}, {entry['Formula']}")
            print(f"{web_url}/compounds/{entry['FAIR_ID']}\n")

dropdown.observe(show_class, names='value')

display(dropdown, output)


Dropdown(description='Family:', layout=Layout(width='300px'), options=('Aminocoumarins', 'Aminoglycosides', 'A…

Output()

<a id="searching"></a>
## Searching & Finding in the AB-DB Database

**Searching** for a particular entry from the **AB-DB collection**.

In [7]:
# Keyword to search for:
keyword = "aminocoumarins"

# Get a list with all compounds from the specified family from the AB-DB collection
simulations_request = api_url + f'/compounds?family={keyword}'

# Query the API
with urllib.request.urlopen(simulations_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

#print(json.dumps(parsed_response['data'], indent=2))

for entry in parsed_response['data']:
    compound = entry['Compound']
    family = entry['Family']
    charge = entry['Charge']
    print(f"{family.title()}-{compound.title()} (Charge: {charge})")
    print(f"{web_url}/compounds/{entry['FAIR_ID']}\n")

Aminocoumarins-Chlorobiocin (Charge: -1)

https://mmb.irbbarcelona.org/ABDB/compounds/ABDB001001

Aminocoumarins-Declovanillobiocin (Charge: -1)

https://mmb.irbbarcelona.org/ABDB/compounds/ABDB001002

Aminocoumarins-Isovanillobiocin (Charge: -1)

...

https://mmb.irbbarcelona.org/ABDB/compounds/ABDB001008

Aminocoumarins-Vanillobiocin (Charge: -1)

https://mmb.irbbarcelona.org/ABDB/compounds/ABDB001009

Aminocoumarins-Coumermycin_A1 (Charge: 0)

https://mmb.irbbarcelona.org/ABDB/compounds/ABDB001010

<a id="descriptors"></a>
## Physico-chemical Descriptors from AB-DB

In [8]:
# Accession to search for:
accession = "ABDB001006"

# Set the descriptors query URL for the API
specific_project_url = api_url + f'/compounds/{accession}'
descriptors_query = specific_project_url + '/files/csv'
print('We query the API at ' + descriptors_query)

# Download the file with an arbitrary name
descriptors_filename = f'{accession}_descriptors.csv'

# Query the API
urllib.request.urlretrieve(descriptors_query, descriptors_filename)

if exists(descriptors_filename):
    print(f'File has been downloaded successfully ({descriptors_filename})')

df = pd.read_csv(descriptors_filename)
df.style.set_table_attributes('class="dataframe table table-striped table-bordered"')

We query the API at https://mmb.irbbarcelona.org/ABDB/api/compounds/ABDB001006/files/csv

File has been downloaded successfully (ABDB001006_descriptors.csv)

,FAIR_ID,Family,Compound,Formula,Smiles,Charge,XLOGP3,GLOB,PBF,RB,Primary_amine,ZWITT,DFT_EN,HOMO,LUMO,GAP,DIP,POL_ISO,POL_ANI,ROT_A,ROT_B,ROT_C,E_TH,CV,S,WAT1,ERR_WA1,WAT2,ERR_WA2,RMSF,ERR_RMSF,ASP,ERR_ASP,ACY,ERR_ACY,K2,ERR_K2,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,NumRadicalElectrons,MaxPartialCharge,MinPartialCharge,MaxAbsPartialCharge,MinAbsPartialCharge,FpDensityMorgan1,FpDensityMorgan2,FpDensityMorgan3,BCUT2D_MWHI,BCUT2D_MWLOW,BCUT2D_CHGHI,BCUT2D_CHGLO,BCUT2D_LOGPHI,BCUT2D_LOGPLOW,BCUT2D_MRHI,BCUT2D_MRLOW,AvgIpc,BalabanJ,BertzCT,Chi0,Chi0n,Chi0v,Chi1,Chi1n,Chi1v,Chi2n,Chi2v,Chi3n,Chi3v,Chi4n,Chi4v,HallKierAlpha,Ipc,Kappa1,Kappa2,Kappa3,LabuteASA,PEOE_VSA1,PEOE_VSA10,PEOE_VSA11,PEOE_VSA12,PEOE_VSA13,PEOE_VSA14,PEOE_VSA2,PEOE_VSA3,PEOE_VSA4,PEOE_VSA5,PEOE_VSA6,PEOE_VSA7,PEOE_VSA8,PEOE_VSA9,SMR_VSA1,SMR_VSA10,SMR_VSA2,SMR_VSA3,SMR_VSA4,SMR_VSA5,SMR_VSA6,SMR_VSA7,SMR_VSA8,SMR_VSA9,SlogP_VSA1,SlogP_VSA10,SlogP_VSA11,SlogP_VSA12,SlogP_VSA2,SlogP_VSA3,SlogP_VSA4,SlogP_VSA5,SlogP_VSA6,SlogP_VSA7,SlogP_VSA8,SlogP_VSA9,TPSA,EState_VSA1,EState_VSA10,EState_VSA11,EState_VSA2,EState_VSA3,EState_VSA4,EState_VSA5,EState_VSA6,EState_VSA7,EState_VSA8,EState_VSA9,VSA_EState1,VSA_EState10,VSA_EState2,VSA_EState3,VSA_EState4,VSA_EState5,VSA_EState6,VSA_EState7,VSA_EState8,VSA_EState9,FractionCSP3,HeavyAtomCount,NHOHCount,NOCount,NumAliphaticCarbocycles,NumAliphaticHeterocycles,NumAliphaticRings,NumAmideBonds,NumAromaticCarbocycles,NumAromaticHeterocycles,NumAromaticRings,NumAtomStereoCenters,NumBridgeheadAtoms,NumHAcceptors,NumHDonors,NumHeteroatoms,NumHeterocycles,NumRotatableBonds,NumSaturatedCarbocycles,NumSaturatedHeterocycles,NumSaturatedRings,NumSpiroAtoms,NumUnspecifiedAtomStereoCenters,Phi,RingCount,MolLogP,MolMR,fr_Al_COO,fr_Al_OH,fr_Al_OH_noTert,fr_ArN,fr_Ar_COO,fr_Ar_N,fr_Ar_NH,fr_Ar_OH,fr_COO,fr_COO2,fr_C_O,fr_C_O_noCOO,fr_C_S,fr_HOCCN,fr_Imine,fr_NH0,fr_NH1,fr_NH2,fr_N_O,fr_Ndealkylation1,fr_Ndealkylation2,fr_Nhpyrrole,fr_SH,fr_aldehyde,fr_alkyl_carbamate,fr_alkyl_halide,fr_allylic_oxid,fr_amide,fr_amidine,fr_aniline,fr_aryl_methyl,fr_azide,fr_azo,fr_barbitur,fr_benzene,fr_benzodiazepine,fr_bicyclic,fr_diazo,fr_dihydropyridine,fr_epoxide,fr_ester,fr_ether,fr_furan,fr_guanido,fr_halogen,fr_hdrzine,fr_hdrzone,fr_imidazole,fr_imide,fr_isocyan,fr_isothiocyan,fr_ketone,fr_ketone_Topliss,fr_lactam,fr_lactone,fr_methoxy,fr_morpholine,fr_nitrile,fr_nitro,fr_nitro_arom,fr_nitro_arom_nonortho,fr_nitroso,fr_oxazole,fr_oxime,fr_para_hydroxylation,fr_phenol,fr_phenol_noOrthoHbond,fr_phos_acid,fr_phos_ester,fr_piperdine,fr_piperzine,fr_priamide,fr_prisulfonamd,fr_pyridine,fr_quatN,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea,accession,inchikey,pubchem
0,ABDB001006,aminocoumarins,plazomicin,C25H53N6O10,O([C@H]1[C@H](NC(=O)[C@@H](O)CC[NH3+])C[C@H]([NH3+])[C@@H](O[C@H]2OC(=CC[C@H]2[NH3+])C[NH2+]CCO)[C@@H]1O)[C@H]1OC[C@](O)([C@H]([NH2+]C)[C@H]1O)C,5,-6.190000,0.069230,1.156960,13,True,False,-2063.494516,-0.663650,-0.481830,0.181820,15.070000,412.780000,55.100000,0.077156,0.055616,0.035499,562.433000,169.178000,252.731000,72.800000,4.100000,143.700000,5.900000,0.169177,0.065881,3.860000,0.540000,2.590000,0.370000,0.270000,0.070000,12.736189,12.736189,0.043707,-1.338023,0.089123,40.390244,597.731000,544.307000,597.379574,238,0,0.252376,-0.457100,0.457100,0.252376,1.219512,1.926829,2.585366,16.712413,9.829801,2.538332,-2.465830,2.305855,-2.957607,5.803673,-0.694869,2.913351,1.788449,868.729118,30.319262,24.138388,24.138388,19.426290,14.455627,14.455627,11.504698,11.504698,8.024739,8.024739,5.671577,5.671577,-1.510000,581610116.026535,33.985690,15.285894,8.535016,239.046896,77.631457,48.645354,24.194999,5.907180,6.290027,0,4.794537,0,0,0,0,12.999757,19.262465,39.392790,77.109206,5.907180,0,5.316789,0,99.054502,39.895705,11.835185,0,0,33.151368,0,0,0,144.203821,23.741989,0,26.186202

<a id="md"></a>
## MD Data from AB-DB

**Downloading** and **visualizing** **MD data** for a particular **AB-DB compound**.

1) **Structure downloading**
2) **Structure visualization**
3) **Trajectory downloading**
4) **Trajectory visualization**
5) **Trajectory analyses (MDDB)**

***
1) **Structure downloading**
***

In [9]:
# Set the structure query URL for the API
specific_project_url = api_url + f'/compounds/{accession}'
structure_query = specific_project_url + '/files/structure.pdb'
print('We query the API at ' + structure_query)

# Download the file with an arbitrary name
structure_filename = f'{accession}_structure.pdb'

# Query the API
urllib.request.urlretrieve(structure_query, structure_filename)

if exists(structure_filename):
    print(f'File has been downloaded successfully ({structure_filename})')

We query the API at https://mmb.irbbarcelona.org/ABDB/api/compounds/ABDB001006/files/structure.pdb

File has been downloaded successfully (ABDB001006_structure.pdb)

***
2) **Structure visualization**
***

In [10]:
view = nglview.show_structure_file(structure_filename)
view._remote_call('setSize', target='Widget', args=['','600px'])
view

NGLWidget()

***
3) **Trajectory downloading**
***

In [11]:
# Set the trajectory query URL for the API
specific_project_url = api_url + f'/compounds/{accession}'
trajectory_query = specific_project_url + '/files/trajectory.xtc'
print('We query the API at ' + trajectory_query)

# Download the file with an arbitrary name
trajectory_filename = f'{accession}_trajectory.xtc'

# Query the API
urllib.request.urlretrieve(trajectory_query, trajectory_filename)

if exists(trajectory_filename):
    print(f'File has been downloaded successfully ({trajectory_filename})')

We query the API at https://mmb.irbbarcelona.org/ABDB/api/compounds/ABDB001006/files/trajectory.xtc

File has been downloaded successfully (ABDB001006_trajectory.xtc)

***
4) **Trajectory visualization**
***

In [14]:
# Show trajectory
try:
    import simpletraj
    view2 = nglview.show_simpletraj(
        nglview.SimpletrajTrajectory(trajectory_filename, structure_filename), gui=True)
    view2._remote_call('setSize', target='Widget', args=['800px','600px'])
    view2
except ImportError:
    import mdtraj as md, os as _os, py3Dmol
    from IPython.display import display as _display

    _t = md.load(trajectory_filename, top=structure_filename)
    _stride = max(1, _t.n_frames // 100)
    _t_sub = _t[::_stride]
    print(f'{_t.n_frames} frames → animating {_t_sub.n_frames} (stride {_stride})')

    _traj_pdb = _os.path.splitext(trajectory_filename)[0] + '_traj.pdb'
    _t_sub.save_pdb(_traj_pdb)
    with open(_traj_pdb) as _f:
        _pdb_content = _f.read()

    view2 = py3Dmol.view(width=800, height=600)
    view2.addModelsAsFrames(_pdb_content)
    view2.setStyle({'stick': {}})
    view2.zoomTo()
    view2.animate({'loop': 'forward', 'reps': 0, 'interval': 100})
    view2.show()


100190 frames → animating 101 (stride 1001)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

***
5) **Trajectory analyses** (MDDB)
***

In [ ]:
# Retrieve the MDDB accession from the specific ABDB compound
mddb_entry_request = api_url + f'/compounds/{accession}'

# Query the API
with urllib.request.urlopen(mddb_entry_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

# Print link to MDDB MD analyses
compound_title = f"{parsed_response['Family'].title()}-{parsed_response['Compound'].title()} ({parsed_response['Charge']})"
mddb_link = f"MDDB (MDposit) link to MD analyses: [link={mddb_web}/#/id/{parsed_response['accession']}]{mddb_web}/#/id/{parsed_response['accession']}[/link]"
print(Panel(f"[bold blue]{compound_title}[/bold blue]\n\n{mddb_link}"))


<a id="qm"></a>
## QM data from AB-DB

**Downloading** and **exploring** **QM data** from the **AB-DB** entries

1) Download the **QM-optimized** structure and compare it against the original one
2) **AMBER force-field parameters** generated from the **QM calculations**
3) **Geometry optimization** and **Single-point energy calculation** data (ioChem-BD)

***
1) Download the **QM-optimized** structure and compare it against the original one
***

In [ ]:
qm_query = f"{api_url}/compounds/{accession}/files/qm_optimized.sdf"

# Download the file with an arbitrary name
qm_filename = f'{accession}_qm_optimized.sdf'

# Query the API
urllib.request.urlretrieve(qm_query, qm_filename)

if exists(qm_filename):
    print(f'File has been downloaded successfully ({qm_filename})')

In [ ]:
view = nglview.show_structure_file(qm_filename)
view._remote_call('setSize', target='Widget', args=['','600px'])
view

In [ ]:
# Show structures: protein, ligand and protein-ligand complex
view1 = nglview.show_structure_file(structure_filename)
view1._remote_call('setSize', target='Widget', args=['550px','400px'])
view1.camera='orthographic'
view1
view2 = nglview.show_structure_file(qm_filename)
view2.add_representation(repr_type='ball+stick')
view2._remote_call('setSize', target='Widget', args=['550px','400px'])
view2.camera='orthographic'
view2
ipywidgets.HBox([view1, view2])

***
2) **AMBER force-field parameters** generated from the **QM calculations**
***

In [ ]:
# AMBER frcmod
ff_frcmod_query = f"{api_url}/compounds/{accession}/files/ff_mol.frcmod"

# Download the file with an arbitrary name
ff_frcmod_filename = f'{accession}_ff.frcmod'

# Query the API
urllib.request.urlretrieve(ff_frcmod_query, ff_frcmod_filename)

if not exists(ff_frcmod_filename):
    print(f'File has not been downloaded successfully ({ff_frcmod_filename}). Please check!')

# AMBER prep file
ff_prep_query = f"{api_url}/compounds/{accession}/files/ff_mol.prep"

# Download the file with an arbitrary name
ff_prep_filename = f'{accession}_ff.prep'

# Query the API
urllib.request.urlretrieve(ff_prep_query, ff_prep_filename)

if not exists(ff_prep_filename):
    print(f'File has not been downloaded successfully ({ff_prep_filename}). Please check!')

frcmod_url = f"/files/{ff_frcmod_filename}"
prep_url = f"/files/{ff_prep_filename}"

print(
    Panel(
        f"[bold]AMBER frcmod file:[/bold]\n\n[link={frcmod_url}]{ff_frcmod_filename}[/link]",
        border_style="grey37"
    ),
    Panel(
        f"[bold]AMBER prep file:[/bold]\n\n[link={prep_url}]{ff_prep_filename}[/link]",
        border_style="grey37"
    )
)

***
3) **Geometry optimization** and **Single-point energy calculation** data (ioChem-BD)
***

In [ ]:
# Retrieve the MDDB accession from the specific ABDB compound
mddb_entry_request = api_url + f'/compounds/{accession}'

# Query the API
with urllib.request.urlopen(mddb_entry_request) as response:
    parsed_response = json.loads(response.read().decode("utf-8"))

family = parsed_response['Family']
compound = parsed_response['Compound']
charge = parsed_response['Charge']

ioChem_url = f"{iochem_bd_web}/simple-search?query=%22{family}%20{compound}%20{charge}%22"

print(
    Panel(
        f"[bold]ioChem-BD QM entries:[/bold]\n\n[link={ioChem_url}]{ioChem_url}[/link]",
        border_style="grey37"
    )
)

***
<a id="questions"></a>

## Questions & Comments

Questions, issues, suggestions and comments are really welcome!

* GitHub issues:
    * [https://github.com/bioexcel/biobb](https://github.com/bioexcel/biobb)